<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

We're going to quickly refactor the pipeline to use pytask instead of hydra and snakemake. This will hopefully demonstrate a simpler and more flexible way to manage data pipelines in Python.

To start off, we need to create a function that queries the CDS API with one job. This function will be used to download the data for each query in the range specified in the data catalog in the config file.

Let's take a look at the data catalog we created in the config module:

You can see the queries entry we created in the data catalog. Each query is a namedtuple that contains the parameters for the CDS API query. The `query` namedtuple has the following variable fields (other fields are singletons): `year`, `month`, `geography`, and `variable`.

In [ ]:
queries = data_catalog['queries'].load()
queries[-3:]

[Query(year='2024', month='11', day=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31'], time=['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00'], geography={'name': 'nepal', 'shapefile': 'https://data.humdata.org/dataset/07db728a-4f0f-4e98-8eb0-8fa9df61f01c/resource/2eb4c47f-fd6e-425d-b623-d35be1a7640e/download/npl_adm_nd_20240314_ab_shp.zip'}, product_type='reanalysis', variables=['2m_dewpoint_temperature', '2m_temperature', 'total_precipitation', 'volumetric_soil_water_layer_1']),
 Query(year='2024', month='12', day=['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31'], time=

We can test this query like we did in the original work:

In [ ]:
example_query = queries[0]

create_bounding_box(example_query.geography['shapefile'])

[np.float64(-11.5), np.float64(42.7), np.float64(-26.1), np.float64(50.9)]

In this way, we have a similar approach as Hydra configs, but, using the `pytask` data catalog, we can more easily gather the data for a specific task in structured manner entirely in Python.

In [ ]:
client = cdsapi.Client()

ex_bounding_box = create_bounding_box(example_query.geography['shapefile'])

request = {
            "product_type": example_query.product_type,
            "variable": example_query.variables, 
            "year": example_query.year,
            "month": example_query.month,
            "day": example_query.day,
            "time": example_query.time,
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": ex_bounding_box
        }

target = f"{example_query.name()}.nc"

client.retrieve("reanalysis-era5-land", request).download(target)

2025-07-29 13:35:42,386 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-07-29 13:35:48,821 INFO Request ID is 48a71608-be3e-41fd-acf7-c0542542bc1e
2025-07-29 13:35:48,965 INFO status has been updated to accepted
2025-07-29 13:35:57,620 INFO status has been updated to running
2025-07-29 13:42:09,192 INFO status has been updated to successful


'2009-1_madagascar.nc'

This works! So now we just need to create a `task_` function that pytask will recognise to parallelise the download of queries over:

Because we defined this task in a function and loop, we can easily debug it by simply calling it:

In [ ]:
task_download_raw_data()

2009-1_nepal


2025-07-29 14:18:21,553 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-07-29 14:18:23,913 INFO Request ID is 4f6ccab4-3236-4850-bce2-38d0abb73c8b
2025-07-29 14:18:24,220 INFO status has been updated to accepted
2025-07-29 14:18:32,862 INFO status has been updated to running
2025-07-29 14:18:38,061 INFO status has been updated to successful


Path('/net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/bld/2009-1_nepal.nc')